##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [9]:
# place your code here
%pip install pinecone

Note: you may need to restart the kernel to use updated packages.


From 'pinecone' library import Pinecone, ServerlessSpec.

In [10]:
# place your code here
from pinecone import (
    Pinecone, 
    DenseVectorQuery
)
from dotenv import load_dotenv
import os
import time

if not os.getenv("PINECONE_API_KEY"):
    os.environ.pop("PINECONE_API_KEY", None)
    
load_dotenv()

True

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

In [11]:
cars = [
    {
        "_id": "car-001",
        "title": "Toyota Corolla",
        "description": "A reliable and fuel-efficient compact sedan designed for everyday driving. Known for its practicality, low maintenance costs and comfortable ride."
    },
    {
        "_id": "car-002",
        "title": "BMW M3",
        "description": "A high-performance sports sedan with a powerful engine, precise handling and sporty design. Built for drivers who enjoy speed and dynamic driving."
    },
    {
        "_id": "car-003",
        "title": "Tesla Model 3",
        "description": "An all-electric sedan featuring modern technology, minimalist interior design and impressive acceleration. Designed for efficient and environmentally friendly transportation."
    },
    {
        "_id": "car-004",
        "title": "Tesla Model S",
        "description": "A premium electric sedan offering powerful acceleration, advanced technology, spacious interior and long-distance driving capabilities."
    },
    {
        "_id": "car-005",
        "title": "Honda NSX",
        "description": "A Japanese mid-engine sports car known for its balanced handling, innovative engineering and combination of performance and everyday usability."
    },
    {
        "_id": "car-006",
        "title": "Mercedes S63",
        "description": "A luxurious high-performance sedan combining a powerful engine, elegant interior, advanced comfort features and sophisticated technology."
    },
    {
        "_id": "car-007",
        "title": "BMW 530d",
        "description": "A diesel-powered executive sedan offering strong torque, comfortable long-distance driving, premium interior quality and balanced performance."
    },
    {
        "_id": "car-008",
        "title": "Kia Ceed",
        "description": "A practical compact hatchback designed for everyday transportation. Offers good fuel economy, comfortable seating and useful interior space."
    },
    {
        "_id": "car-009",
        "title": "Ferrari F50",
        "description": "An Italian limited-production supercar featuring a naturally aspirated V12 engine, mid-engine layout and racing-inspired engineering."
    },
    {
        "_id": "car-010",
        "title": "Bugatti Veyron",
        "description": "An extraordinary high-performance hypercar featuring a quad-turbocharged W16 engine, all-wheel drive and exceptional top speed."
    }
]

When you perform "Search", use of your own text string.

In [14]:
# place your code here
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
model = "multilingual-e5-large"
model_info = pc.inference.model.get(model)
dimension = model_info.default_dimension

pc.indexes.create(
    name="carsindex",
    schema={"fields": {"your_vector": {"type": "dense_vector", "dimension": dimension, "metric": "cosine"}}},
    deployment={"deployment_type": "managed", "cloud": "aws", "region": "us-east-1"},
)

while not pc.indexes.describe("carsindex").status.ready:
    time.sleep(1)

index = pc.index("carsindex")

texts = [car["description"] for car in cars]

embeddings = pc.inference.embed(
    model = "multilingual-e5-large",
    inputs=texts,
    parameters={"input_type": "passage"}
)

documents = []

for car, embedding in zip(cars, embeddings.data):
    documents.append({
        "_id": car["_id"],
        "your_vector": embedding.values,
        "title": car["title"],
        "description": car["description"]
    })

index.documents.upsert(
    namespace="cars",
    documents=documents
)

query = "I want a car that can fit 5 people comfortably."

query_embedding = pc.inference.embed(
    model=model,
    inputs=[query],
    parameters={"input_type": "query"}
)

results = index.documents.search(
    namespace="cars",
    top_k=3,
    score_by=[DenseVectorQuery(field="your_vector", values = query_embedding.data[0].values)],
    include_fields=["title", "description"],
)

Print search results.

In [16]:
print("\nSearch query:", query)
print("\nMost relevant cars:\n")

for match in results.matches:
    print("ID:", match.id)
    print("Score:", match.score)
    print("Title:", match.title)
    print("Description:", match.description)
    print("-" * 50)


Search query: I want a car that can fit 5 people comfortably.

Most relevant cars:

ID: car-006
Score: 0.8263211
Title: Mercedes S63
Description: A luxurious high-performance sedan combining a powerful engine, elegant interior, advanced comfort features and sophisticated technology.
--------------------------------------------------
ID: car-001
Score: 0.8246414
Title: Toyota Corolla
Description: A reliable and fuel-efficient compact sedan designed for everyday driving. Known for its practicality, low maintenance costs and comfortable ride.
--------------------------------------------------
ID: car-007
Score: 0.82452035
Title: BMW 530d
Description: A diesel-powered executive sedan offering strong torque, comfortable long-distance driving, premium interior quality and balanced performance.
--------------------------------------------------
